In [1]:
# =====================================================================
# Kaggle: train YOLO26 on Merged_Dataset (0 = person, 1 = robotic_arm)
# Notebook settings: Accelerator = "GPU T4 x2" (or P100), Internet = ON
# Add your uploaded dataset via "Add Input" before running.
# Paste each "# %% " block into its own notebook cell (or run all as one).
# =====================================================================

In [2]:
!pip install -q "ultralytics==8.4.104"

In [3]:
import glob, os, yaml, torch

hits = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert hits, "data.yaml not found - did you add the dataset with 'Add Input'?"
DATA_ROOT = os.path.dirname(hits[0])
print("Dataset root:", DATA_ROOT)

cfg = {
    "path": DATA_ROOT,          # /kaggle/input is read-only, so the yaml goes to /kaggle/working
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": {0: "person", 1: "robotic_arm"},
}
for split in ("train", "valid", "test"):
    n = len(os.listdir(f"{DATA_ROOT}/{split}/images"))
    print(f"{split}: {n} images")

DATA_YAML = "/kaggle/working/data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

Dataset root: /kaggle/input/datasets/kumarcom/hruman/Merged_Dataset
train: 51955 images
valid: 8994 images
test: 1611 images


In [4]:
from ultralytics import YOLO

n_gpu = torch.cuda.device_count()
DEVICE = list(range(n_gpu)) if n_gpu > 1 else (0 if n_gpu == 1 else "cpu")
print("GPUs:", n_gpu, "->", DEVICE)

PROJECT, NAME = "/kaggle/working/runs", "merged_yolo26s"
last = f"{PROJECT}/{NAME}/weights/last.pt"

if os.path.exists(last):                 # session restarted: continue where it stopped
    model = YOLO(last)
    model.train(resume=True)
else:
    model = YOLO("yolo26s.pt")           # downloads pretrained COCO weights
    model.train(
        data=DATA_YAML,
        epochs=30,
        imgsz=640,
        batch=32,                        # total across GPUs; lower to 16 if you get CUDA OOM
        device=DEVICE,
        workers=4,                       # Kaggle gives 4 CPU cores
        cache=False,                     # dataset is too big for Kaggle RAM/disk cache
        patience=10,                     # stop early if val mAP stalls for 10 epochs
        save_period=5,                   # extra checkpoint every 5 epochs
        cos_lr=True,
        close_mosaic=5,                  # turn off mosaic for the last 5 epochs
        seed=0,
        project=PROJECT,
        name=NAME,
        exist_ok=True,
        plots=True,
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPUs: 2 -> [0, 1]
New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.104 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5,

In [5]:
best = YOLO(f"{PROJECT}/{NAME}/weights/best.pt")
m = best.val(data=DATA_YAML, split="test", imgsz=640, batch=32, device=0 if n_gpu else "cpu",
             project=PROJECT, name=NAME + "_test", exist_ok=True)
print(f"TEST  mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}")
for i, c in m.names.items():
    print(f"  {c:12s} AP50={m.box.ap50[i]:.4f}  AP50-95={m.box.ap[i]:.4f}")

Ultralytics 8.4.104 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 0.5±0.2 MB/s, size: 32.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/kumarcom/hruman/Merged_Dataset/test/labels... 1611 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1611/1611 17.6it/s 1:32
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/kumarcom/hruman/Merged_Dataset/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.2s/it 1:01
                   all       1611       4170      0.848      0.788       0.85      0.659
                person       1151       3683      0.769      0.623       0.72      0.453
   

In [6]:
import shutil
shutil.make_archive("/kaggle/working/merged_yolo26s_results", "zip", PROJECT)
print("Download: /kaggle/working/merged_yolo26s_results.zip  (best.pt is inside weights/)")

Download: /kaggle/working/merged_yolo26s_results.zip  (best.pt is inside weights/)
